# 02.03 — Quantization & llama-cpp-python

**Tujuan**: pakai SLM yang lebih besar (TinyLlama 1.1B) di CPU laptop tanpa OOM, lewat **quantization 4-bit**. Bandingkan ukuran disk, RAM, latency dengan SmolLM2-135M fp32.

**Prasyarat**: notebook 02.02 lulus.

**Model**: `TinyLlama-1.1B-Chat-v1.0` quantized ke **Q4_K_M GGUF** (~670 MB di disk). Dijalankan lewat `llama-cpp-python`.

> Untuk internals quantization (kenapa Q4_K_M, bagaimana kalibrasi int4, dll), lihat [`llm-internals/05`](../../llm-internals/) modul 05.

## 0. Bootstrap (jalankan pertama)

In [ ]:
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_NAME = "llm-vs-slm-lab"
    REPO_URL = "https://github.com/rizkyhaksono/llm-vs-slm-lab.git"
    if not Path(REPO_NAME).exists():
        !git clone {REPO_URL}
    %cd {REPO_NAME}
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu
    !pip install -q -r requirements.txt

repo_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "requirements.txt").exists():
        repo_root = candidate
        break
assert repo_root is not None
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f"IN_COLAB={IN_COLAB}, repo_root={repo_root}")

## 1. Apa itu quantization?

Model dilatih dengan **weight floating-point 32-bit** (fp32) — tiap angka pakai 4 byte. Untuk model 1.1B params, ini berarti **~4.4 GB** RAM cuma untuk weights.

Quantization = kompres tiap weight jadi **lebih sedikit bit**:

| Precision | Bit per weight | Ukuran 1.1B model | Quality drop |
|---|---|---|---|
| fp32 | 32 | ~4.4 GB | baseline |
| fp16 | 16 | ~2.2 GB | minimal |
| int8 | 8 | ~1.1 GB | sedikit |
| **Q4_K_M** | ~4.5 (mixed) | **~670 MB** | **sedang** — paling balance |
| Q2 | 2 | ~340 MB | besar (sering ngawur) |

Q4_K_M = standar de-facto untuk "kecil tapi masih oke". Sesuai filosofi repo ini: realistis di CPU laptop.

## 2. Download GGUF

Format **GGUF** = single-file untuk model quantized, di-baca oleh `llama.cpp` (C++ engine). Self-contained: weights + tokenizer + config dalam 1 file.

In [ ]:
from huggingface_hub import hf_hub_download

MODEL_REPO = "TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF"
MODEL_FILE = "tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf"

print(f"Downloading {MODEL_FILE} (~670 MB)... pertama kali bisa 2-5 menit")
model_path = hf_hub_download(
    repo_id=MODEL_REPO,
    filename=MODEL_FILE,
    local_dir=str(repo_root / "models"),
)

size_mb = Path(model_path).stat().st_size / (1024 * 1024)
print(f"\nDownloaded ke: {model_path}")
print(f"Ukuran:        {size_mb:.0f} MB")

## 3. Load model dengan llama-cpp-python

In [ ]:
from llama_cpp import Llama

llm = Llama(
    model_path=model_path,
    n_ctx=2048,          # context window (max input + output tokens)
    n_threads=4,         # CPU threads — sesuaikan dengan core mu
    verbose=False,       # supaya tidak banjir log
)

print("OK — TinyLlama Q4 loaded.")

## 4. Generate dengan chat-style prompt

`llama-cpp-python` punya method `.create_chat_completion()` yang **OpenAI-compatible** — pattern messages sama persis seperti Groq di notebook 02.01.

In [ ]:
import time

def chat_tinyllama(prompt: str, max_tokens: int = 80, temperature: float = 0.0) -> tuple[str, float]:
    t0 = time.perf_counter()
    resp = llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=temperature,
    )
    elapsed_ms = (time.perf_counter() - t0) * 1000
    text = resp["choices"][0]["message"]["content"].strip()
    return text, elapsed_ms

text, ms = chat_tinyllama("Apa ibu kota Indonesia? Jawab dalam 1 kalimat.")
print(f"Response: {text}")
print(f"Latency:  {ms:.0f} ms")

## 5. Bandingkan ukuran & kecepatan: SmolLM2-135M fp32 vs TinyLlama-1.1B Q4

Insight kunci: **model 8x lebih besar (params) bisa lebih cepat & ukuran di-disk-nya lebih kecil** lewat quantization.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load SmolLM2 fp32 (same as notebook 02.02)
smol_id = "HuggingFaceTB/SmolLM2-135M-Instruct"
smol_tok = AutoTokenizer.from_pretrained(smol_id)
smol_model = AutoModelForCausalLM.from_pretrained(smol_id, torch_dtype=torch.float32)
smol_model.eval()

def chat_smollm(prompt: str, max_new_tokens: int = 80) -> tuple[str, float]:
    inputs = smol_tok.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True, return_tensors="pt", return_dict=True,
    )
    t0 = time.perf_counter()
    with torch.no_grad():
        out = smol_model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=smol_tok.eos_token_id,
        )
    ms = (time.perf_counter() - t0) * 1000
    text = smol_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text.strip(), ms

prompt = "Tuliskan 3 fakta menarik tentang Komodo (hewan)."

smol_text, smol_ms = chat_smollm(prompt, max_new_tokens=100)
tiny_text, tiny_ms = chat_tinyllama(prompt, max_tokens=100)

print("=" * 70)
print(f"SmolLM2-135M fp32 ({smol_ms:.0f} ms):")
print(smol_text)
print("=" * 70)
print(f"TinyLlama-1.1B Q4 ({tiny_ms:.0f} ms):")
print(tiny_text)
print("=" * 70)

In [ ]:
# Tabel summary
import pandas as pd

summary = pd.DataFrame([
    {"model": "SmolLM2-135M", "params": "135M", "precision": "fp32", "disk_mb": 270, "latency_ms": round(smol_ms)},
    {"model": "TinyLlama-1.1B", "params": "1.1B", "precision": "Q4_K_M", "disk_mb": round(size_mb), "latency_ms": round(tiny_ms)},
])
summary

## Refleksi & insight

1. **Quantization = magic untuk CPU**. 8x lebih banyak params di Q4 nggak otomatis 8x lebih lambat — bahkan sering lebih cepat, karena memory bandwidth bukan compute yang jadi bottleneck di CPU.
2. **Q4_K_M sweet spot**: turunkan ukuran 6-7x dari fp32, quality drop ringan untuk task umum. Untuk math/code, drop bisa significant — selalu A/B test.
3. **Bahasa Indonesia di TinyLlama**: masih terbatas (training English-dominant), tapi lebih natural dari SmolLM2-135M.
4. **GGUF + llama.cpp** standar de-facto untuk run LLM di CPU/edge device. Ekosistem-nya besar (Mac, Windows, Linux, Android, iOS, browser via WASM).

## Cross-link

Mau paham internals quantization (algoritma int4, calibration, why Q4_K_M)? → [`llm-internals`](../../llm-internals/) modul 05.

## Latihan mandiri

1. Coba download GGUF lain dari TheBloke, mis. `phi-2.Q4_K_M.gguf` (Phi-2 2.7B). Bandingkan kualitas Bahasa-nya dengan TinyLlama.
2. Coba `temperature=0.7` vs `temperature=0.0` di TinyLlama — apa beda kualitas response untuk task kreatif (mis. "buatkan puisi singkat")?

## Lanjut

Sekarang gabungkan semua jadi benchmark formal LLM vs SLM (fp32) vs SLM (Q4): [04_benchmark_side_by_side.ipynb](04_benchmark_side_by_side.ipynb)